In [ ]:
# SPR 2026 - XLM-RoBERTa Large + Focal Loss + Optuna Calibration (modelo proprio)
# XLM-R Large: 550M params, treinado em 100 linguas com dados massivos.
# Forte em portugues apesar de multilingual. Batch menor por ser grande.

import os
import re
import gc
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings
warnings.filterwarnings('ignore')

# =========================================================================
# CONFIG
# =========================================================================
SEED = 42
MAX_LEN = 512
BATCH_SIZE = 4  # XLM-R Large needs small batch on T4
GRAD_ACCUM = 4  # effective batch = 16
EPOCHS = 5
LR = 1e-5  # lower LR for larger model
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
N_FOLDS = 5
NUM_CLASSES = 7
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25
N_TRIALS = 300
LABEL_SMOOTHING = 0.05

MODEL_CANDIDATES = [
    '/kaggle/input/models/FacebookAI/xlm-roberta-large/pytorch/default/1',
    '/kaggle/input/xlm-roberta-large',
]

DATA_DIR = '/kaggle/input/competitions/spr-2026-mammography-report-classification'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


def find_model_path():
    for candidate in MODEL_CANDIDATES:
        if os.path.isdir(candidate) and os.path.exists(os.path.join(candidate, 'config.json')):
            return candidate
    # fallback: busca recursiva
    base = '/kaggle/input'
    for root, dirs, files in os.walk(base):
        if 'config.json' in files and 'xlm-roberta-large' in root.lower():
            return root
    return None


MODEL_PATH = find_model_path()


def seed_everything(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)

print('=' * 60)
print('SPR 2026 - XLM-RoBERTa Large + Focal Loss + Optuna')
print('=' * 60)
print(f'Device: {DEVICE}')
print(f'Model: {MODEL_PATH}')
print(f'Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM} effective')
print(f'Epochs: {EPOCHS}, LR: {LR}, MAX_LEN: {MAX_LEN}')

In [ ]:
# =========================================================================
# PREPROCESSING (marcadores com acento)
# =========================================================================
INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']


def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()


def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()


def build_input_text(report_text):
    report      = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)


# Carregar dados
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')

train_df['text'] = train_df['report'].apply(build_input_text)
test_df['text']  = test_df['report'].apply(build_input_text)

print(f'Train: {len(train_df)}, Test: {len(test_df)}')
print(train_df['target'].value_counts().sort_index())

In [ ]:
# =========================================================================
# DATASET + FOCAL LOSS
# XLM-RoBERTa NAO usa token_type_ids
# =========================================================================
class MammographyDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        # XLM-RoBERTa nao retorna token_type_ids, apenas input_ids e attention_mask
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


class FocalLoss(nn.Module):
    """Focal Loss com label smoothing opcional."""

    def __init__(self, alpha=0.25, gamma=2.0, label_smoothing=0.0, num_classes=7):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        self.num_classes = num_classes

    def forward(self, inputs, targets):
        ce = F.cross_entropy(
            inputs, targets,
            reduction='none',
            label_smoothing=self.label_smoothing,
        )
        pt = torch.exp(-ce)
        focal_loss = self.alpha * (1.0 - pt) ** self.gamma * ce
        return focal_loss.mean()


print('Dataset e FocalLoss definidos.')
print('XLM-RoBERTa: sem token_type_ids, apenas input_ids + attention_mask.')

In [ ]:
# =========================================================================
# TRAINING HELPERS (com gradient accumulation)
# =========================================================================
def train_epoch(model, loader, optimizer, scheduler, scaler, criterion, grad_accum=GRAD_ACCUM):
    """Treina uma epoch com gradient accumulation e mixed precision."""
    model.train()
    total_loss = 0.0
    num_batches = 0
    optimizer.zero_grad()

    for step, batch in enumerate(loader):
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['labels'].to(DEVICE)

        with autocast(dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            loss = loss / grad_accum  # normalizar pelo numero de acumulacoes

        scaler.scale(loss).backward()

        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        num_batches += 1

    return total_loss / num_batches


@torch.no_grad()
def evaluate(model, loader):
    """Avalia modelo, retorna probs e labels."""
    model.eval()
    all_probs = []
    all_labels = []

    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)

        with autocast(dtype=torch.float16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        probs = F.softmax(outputs.logits.float(), dim=-1).cpu().numpy()
        all_probs.append(probs)

        if 'labels' in batch:
            all_labels.append(batch['labels'].numpy())

    all_probs = np.vstack(all_probs)
    if all_labels:
        all_labels = np.concatenate(all_labels)
    else:
        all_labels = None
    return all_probs, all_labels


print('Training helpers definidos.')
print(f'Gradient accumulation: {GRAD_ACCUM} steps (effective batch = {BATCH_SIZE * GRAD_ACCUM})')

In [ ]:
# =========================================================================
# CALIBRATION FUNCTIONS
# =========================================================================
def temperature_scale(probs, temperature):
    """Aplica temperature scaling nas probabilidades."""
    logits     = np.log(probs + 1e-10)
    scaled     = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)


def apply_thresholds(probs, thresholds):
    """Aplica threshold multipliers por classe."""
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)


print('Calibration functions definidas.')

In [ ]:
# =========================================================================
# 5-FOLD TRAINING
# =========================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
criterion = FocalLoss(
    alpha=FOCAL_ALPHA,
    gamma=FOCAL_GAMMA,
    label_smoothing=LABEL_SMOOTHING,
    num_classes=NUM_CLASSES,
)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_probs  = np.zeros((len(train_df), NUM_CLASSES))
test_probs = np.zeros((len(test_df), NUM_CLASSES))
fold_scores = []

test_ds = MammographyDataset(
    test_df['text'].values, None, tokenizer, MAX_LEN
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=0, pin_memory=True,
)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df['text'], train_df['target'])):
    print(f'\n{"=" * 60}')
    print(f'FOLD {fold + 1}/{N_FOLDS}')
    print(f'{"=" * 60}')

    seed_everything(SEED + fold)

    # Datasets
    train_ds = MammographyDataset(
        train_df['text'].iloc[train_idx].values,
        train_df['target'].iloc[train_idx].values,
        tokenizer, MAX_LEN,
    )
    val_ds = MammographyDataset(
        train_df['text'].iloc[val_idx].values,
        train_df['target'].iloc[val_idx].values,
        tokenizer, MAX_LEN,
    )

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=0, pin_memory=True, drop_last=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
        num_workers=0, pin_memory=True,
    )

    # Model
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_PATH,
        num_labels=NUM_CLASSES,
        local_files_only=True,
    ).to(DEVICE)

    # Optimizer com weight decay
    no_decay = ['bias', 'LayerNorm.weight', 'LayerNorm.bias']
    param_groups = [
        {
            'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
            'weight_decay': WEIGHT_DECAY,
        },
        {
            'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
            'weight_decay': 0.0,
        },
    ]
    optimizer = torch.optim.AdamW(param_groups, lr=LR)

    # Scheduler: numero de optimizer steps (com grad accum)
    total_steps = (len(train_loader) // GRAD_ACCUM + 1) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    scaler = GradScaler()

    best_f1 = 0.0
    best_state = None

    for epoch in range(EPOCHS):
        train_loss = train_epoch(
            model, train_loader, optimizer, scheduler, scaler, criterion, GRAD_ACCUM
        )

        val_p, val_l = evaluate(model, val_loader)
        val_preds = val_p.argmax(axis=1)
        val_f1 = f1_score(val_l, val_preds, average='macro')

        print(f'  Epoch {epoch + 1}/{EPOCHS} | Loss: {train_loss:.4f} | Val F1: {val_f1:.5f}')

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Carregar melhor checkpoint
    model.load_state_dict(best_state)
    model.to(DEVICE)

    # OOF predictions
    val_p, val_l = evaluate(model, val_loader)
    oof_probs[val_idx] = val_p
    fold_f1 = f1_score(val_l, val_p.argmax(axis=1), average='macro')
    fold_scores.append(fold_f1)
    print(f'  Best Val F1: {fold_f1:.5f}')

    # Test predictions
    test_p, _ = evaluate(model, test_loader)
    test_probs += test_p / N_FOLDS

    # Limpar memoria
    del model, optimizer, scheduler, scaler, best_state
    gc.collect()
    torch.cuda.empty_cache()

print(f'\n{"=" * 60}')
print(f'OOF F1 por fold: {[round(s, 5) for s in fold_scores]}')
print(f'OOF F1 medio: {np.mean(fold_scores):.5f} (+/- {np.std(fold_scores):.5f})')
oof_baseline = f1_score(train_df['target'].values, oof_probs.argmax(axis=1), average='macro')
print(f'OOF F1 global (argmax): {oof_baseline:.5f}')

In [ ]:
# =========================================================================
# OPTUNA CALIBRATION: temperatura + threshold multipliers
# =========================================================================
oof_labels = train_df['target'].values


def objective(trial):
    temp = trial.suggest_float('temperature', 0.3, 3.0)
    thresholds = [
        trial.suggest_float(f't{c}', 0.05, 3.0) for c in range(NUM_CLASSES)
    ]
    calibrated = temperature_scale(oof_probs, temp)
    preds = apply_thresholds(calibrated, thresholds)
    return f1_score(oof_labels, preds, average='macro')


study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

best_temp = study.best_params['temperature']
best_thresholds = [study.best_params[f't{c}'] for c in range(NUM_CLASSES)]

print(f'\nMelhor F1 OOF (Optuna, {N_TRIALS} trials): {study.best_value:.5f}')
print(f'Temperatura: {best_temp:.4f}')
print(f'Thresholds: {[round(t, 4) for t in best_thresholds]}')

# Comparar com baseline
print(f'\nGanho sobre argmax: {study.best_value - oof_baseline:+.5f}')

In [ ]:
# =========================================================================
# SUBMISSION
# =========================================================================
calibrated_test = temperature_scale(test_probs, best_temp)
predictions = apply_thresholds(calibrated_test, best_thresholds)

submission = pd.DataFrame({
    'ID': test_df['ID'],
    'target': predictions,
})
submission.to_csv('/kaggle/working/submission.csv', index=False)

print('=== SUBMISSION ===')
print(submission.to_string(index=False))
print(f'\nsubmission.csv salvo ({len(submission)} linhas)')
print(f'\nDistribuicao das classes:')
print(submission['target'].value_counts().sort_index())